# Introduction & Security Context
Welcome team. This notebook walks through the agent-identity reference implementation for deploying a Google Agent Development Kit (ADK) agent.

When designing cloud-native applications in strict regulatory environments, managing long-lived credentials is a persistent compliance challenge. This implementation demonstrates a paradigm shift: eliminating service-account JSON keys in favor of Vertex AI Agent Engine Agent Identity. By assigning a per-agent SPIFFE principal, we ensure that the agent authenticates securely and natively, adhering to strict least-privilege principles.

## Key Objectives:
1. Understand the Vertex AI Agent Identity architecture.
2. Deploy an ADK agent that securely retrieves internal policy documents from Google Cloud Storage (GCS).
3. Implement keyless authentication for A2A (Application-to-Application) invocations.
4. Register the agent as a Gemini Enterprise add-on.
5. Logging and Monitoring of the Agent in Agent engine

### Prerequisites
1. Enable APIs: `aiplatform.googleapis.com`, `iam.googleapis.com`.
2. Ensure you have the `Project IAM Admin` role to grant permissions to the agent.

In [ ]:
# 1. Install Dependencies
!pip install --upgrade google-cloud-aiplatform google-adk

### Step 1 - Step 1 — Clone and install

In [ ]:
# Clone the repository
!git clone https://github.com/avnit/agent-identity.git
!cd agent-identity

# Setup the Python virtual environment
!python -m venv .venv
!source .venv/bin/activate

# Install dependencies
!pip install -e .
!cp .env.example .env

SyntaxError: invalid syntax (4230791996.py, line 2)

### Step 2 — Create GCS buckets and seed policies

In [ ]:
# 2. Authenticate and Initialize
from google.colab import auth
import vertexai

auth.authenticate_user()

PROJECT_ID = "your-project-id"  # @param {type:"string"}
LOCATION = "us-central1"        # @param {type:"string"}
GOOGLE_CLOUD_PROJECT = "your-project-id"  # @param {type:"string"}
!gcloud config set project {GOOGLE_CLOUD_PROJECT}
STAGING_BUCKET = "gs://your-staging-bucket"  # @param {type:"string"}
POLICY_BUCKET = "gs://your-policy-bucket"    # @param {type:"string"}
!gsutil mb -l {LOCATION} {STAGING_BUCKET}
!gsutil mb -l {LOCATION} {POLICY_BUCKET}

vertexai.init(project=PROJECT_ID, location=LOCATION)
print(f"Initialized project {PROJECT_ID} in {LOCATION}")

Enable required APIs


In [ ]:
# Enable required APIs
gcloud services enable \
  aiplatform.googleapis.com \
  storage.googleapis.com \
  discoveryengine.googleapis.com

# Create buckets with Uniform Bucket-Level Access
gcloud storage buckets create gs://$STAGING_BUCKET \
  --location=$GOOGLE_CLOUD_LOCATION \
  --uniform-bucket-level-access

gcloud storage buckets create gs://$POLICY_BUCKET \
  --location=$GOOGLE_CLOUD_LOCATION \
  --uniform-bucket-level-access

# Seed the environment with sample policies
gcloud storage cp sample_policies/*.md gs://$POLICY_BUCKET/policies/

### Step 3 — Deploy to Vertex AI Agent Engine with Agent Identity

In [ ]:
# Execute the deployment script
python deploy.py

Deployment takes 5–15 minutes while Agent Engine builds the container.

Output you care about:

reasoningEngine — full resource path (copy it)
After deploy, read the agent's effectiveIdentity — this becomes the principal://... string you grant IAM to
AGENT_PRINCIPAL='principal://agents.global.org-<ORG_ID>.system.id.goog/...'

In [ ]:
RESONING_ENGINE="" # @param {type:"string"}
AGENT_PRINCIPAL="" # @param {type:"string"}
curl -s \
  -H "Authorization: Bearer $(gcloud auth application-default print-access-token)" \
  "https://$GOOGLE_CLOUD_LOCATION-aiplatform.googleapis.com/v1beta1/$REASONING_ENGINE" \
  | python -c 'import sys,json; d=json.load(sys.stdin); print("principal://" + d["spec"]["effectiveIdentity"])'

### Step 4 - Grant Permissions to the Identity

In [ ]:

# Bind IAM directly to the agent principal
gcloud storage buckets add-iam-policy-binding gs://$POLICY_BUCKET \
  --member="$AGENT_PRINCIPAL" \
  --role="roles/storage.objectViewer"

In [ ]:
# Testing
python remote_test.py "What's our remote work policy on equipment stipend?"

### Step 5 - Deploy to Gemini Enterpise

In [ ]:
export GEMINI_ENTERPRISE_APP_ID=<your-app-id> # @param {type:"string"}
export GEMINI_ENTERPRISE_LOCATION=global           # or us / eu
export REASONING_ENGINE=projects/<NUM>/locations/<LOC>/reasoningEngines/<ID> # @param {type: "string"}

python register_gemini_enterprise.py

### Step 6 - Test the policy administrator

Sample prompt : Tell me about work from home policy?